# 05. 새 대화에서도 선호를 기억하는 LangGraph

**대상:** Python 함수는 알지만 LangGraph의 기억은 처음인 개발자. **학습 목표:** thread 상태와 장기 기억을 구분합니다.
기본 실습에는 API 키·LLM·임베딩 모델이 필요 없습니다. 패키지 최초 설치에는 인터넷이 필요합니다.
기존 노트북 01–04를 실행하지 않아도 됩니다. 이 노트북 파일 하나에 모든 코드가 들어 있습니다.

## 준비
Python 3.12 이상에서 다음 명령을 실행한 뒤 해당 환경의 커널을 선택하세요.

```bash
uv venv .venv
uv pip install --python .venv/bin/python "memtomem[langgraph]==0.6.0" jupyterlab ipykernel
uv run --python .venv/bin/python --no-project jupyter lab
```

Windows는 `.venv/bin/python` 대신 `.venv/Scripts/python.exe`를 사용합니다.
별도 `nest-asyncio`, ONNX, LangChain 모델 어댑터는 필요하지 않습니다.
아래에서 **Run All**을 실행하세요. 출력은 결정적 템플릿이며 LLM 답변이 아닙니다.

In [ ]:
try:
    import langgraph
    import memtomem
except (ImportError, ModuleNotFoundError) as exc:
    raise RuntimeError(
        '먼저 uv pip install "memtomem[langgraph]==0.6.0" jupyterlab ipykernel 실행 후 '
        '해당 환경의 Python 커널을 선택하세요.'
    ) from exc


## 1. 실습 저장소 격리

실제 홈·설정·API 키를 읽는 기본 경로 대신 임시 홈을 사용합니다. 종료와 예외 시 환경을 복구하고 실습 파일만 정리합니다.
커널을 강제로 중단하면 정리 코드가 실행되지 않을 수 있으므로 다시 시작 후 Run All을 사용하세요.

In [ ]:
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def isolated_lab():
    """실제 홈/클라이언트 설정 대신 이 실습만의 임시 상태를 사용합니다."""
    original_env = dict(os.environ)
    original_cwd = Path.cwd()
    with tempfile.TemporaryDirectory(prefix="memtomem-beginner-") as directory:
        root = Path(directory).resolve()
        # 프록시·사내 CA 환경에서도 선택형 LLM 셀이 동작하도록 네트워크 설정만 남깁니다.
        allowed = {
            "PATH", "LANG", "LC_ALL", "SYSTEMROOT", "WINDIR", "TMPDIR",
            "HTTP_PROXY", "HTTPS_PROXY", "NO_PROXY",
            "http_proxy", "https_proxy", "no_proxy",
            "SSL_CERT_FILE", "SSL_CERT_DIR", "REQUESTS_CA_BUNDLE",
        }
        clean = {k: v for k, v in original_env.items() if k in allowed}
        clean.update(
            HOME=str(root), USERPROFILE=str(root),
            XDG_CONFIG_HOME=str(root / "config"),
            XDG_DATA_HOME=str(root / "data"),
            XDG_STATE_HOME=str(root / "state"),
            XDG_CACHE_HOME=str(root / "cache"),
            MEMTOMEM_FASTEMBED_CACHE=str(root / "cache" / "models"),
            LANGSMITH_TRACING="false", LANGCHAIN_TRACING_V2="false",
        )
        try:
            os.environ.clear()
            os.environ.update(clean)
            os.chdir(root)
            yield root
        finally:
            os.chdir(original_cwd)
            os.environ.clear()
            os.environ.update(original_env)


## 2. 그래프와 두 종류의 기억

`State`는 한 대화의 방문 횟수입니다. `InMemorySaver`는 thread별 상태를 보관하지만 프로세스가 종료되면 사라집니다.
`Context.user_id`로 선택하는 tuple namespace는 여러 thread가 명시적으로 공유할 기억 주소입니다.
`MemtomemBaseStore`는 `compile(store=...)`용 표준 Store이며 저장 원본은 JSON 파일입니다.
Core의 Markdown·SQLite/BM25/RRF 경로와 같지 않고, TTL도 지원하지 않습니다.
`embedding.provider=none`에서 검색은 어휘 겹침 점수이며 의미 번역을 하지 않습니다.

그래프는 **명시적 선호 입력 저장 → 저장된 선호를 읽어 응답 구성** 순서입니다.

In [ ]:
from dataclasses import dataclass
from operator import add
from typing import Annotated, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from memtomem.config import EmbeddingConfig
from memtomem.integrations.langgraph import MemtomemBaseStore

class State(TypedDict, total=False):
    preference: str
    answer: str
    visits: Annotated[int, add]

@dataclass
class Context:
    user_id: str

def make_graph(store):
    async def remember(state: State, runtime: Runtime[Context]):
        # 애플리케이션이 선택한 namespace입니다. 인증/인가 기능은 아닙니다.
        namespace = ("users", runtime.context.user_id)
        if state.get("preference"):
            await runtime.store.aput(namespace, "preferences",
                                     {"text": state["preference"]})
        return {"preference": ""}  # 다음 호출에서 이전 입력을 다시 쓰지 않음

    async def respond(state: State, runtime: Runtime[Context]):
        item = await runtime.store.aget(
            ("users", runtime.context.user_id), "preferences"
        )
        text = item.value["text"] if item else "아직 저장한 선호 없음"
        return {"answer": text, "visits": 1}

    builder = StateGraph(State, context_schema=Context)
    builder.add_node("remember", remember)
    builder.add_node("respond", respond)
    builder.add_edge(START, "remember")
    builder.add_edge("remember", "respond")
    builder.add_edge("respond", END)
    return builder.compile(checkpointer=InMemorySaver(), store=store)


## 3. 비교 실험

입력은 가상의 선호 한 문장입니다. 같은 thread의 방문 횟수는 증가하고 새 thread는 1부터 시작해야 합니다.
같은 사용자는 선호를 재사용하지만 다른 사용자는 읽지 않습니다. namespace 선택은 인증/인가를 대신하지 않습니다.
동일 key를 업데이트하면 새 기억을 무한히 추가하는 대신 한 파일을 갱신합니다.

In [ ]:
PREFERENCE = "설명은 한국어로, 코드 식별자는 원문 그대로."
UPDATED = "설명은 한국어로 짧게, 코드 식별자는 원문 그대로."

async def demonstrate(root):
    store_root = root / "records"
    embedding = EmbeddingConfig(provider="none", dimension=0)
    store = MemtomemBaseStore(root=store_root, embedding=embedding)
    try:
        graph = make_graph(store)
        a = Context(user_id="learner-a")
        b = Context(user_id="learner-b")
        t1 = {"configurable": {"thread_id": "conversation-1"}}
        t2 = {"configurable": {"thread_id": "conversation-2"}}
        t3 = {"configurable": {"thread_id": "conversation-3"}}

        assert await store.aget(("users", a.user_id), "preferences") is None
        first = await graph.ainvoke({"preference": PREFERENCE}, t1, context=a)
        second = await graph.ainvoke({}, t1, context=a)
        assert first["visits"] == 1 and second["visits"] == 2
        print("PASS 같은 thread: 방문 횟수 1 → 2")

        fresh = await graph.ainvoke({}, t2, context=a)
        assert fresh["visits"] == 1 and fresh["answer"] == PREFERENCE
        print("PASS 새 thread: 대화는 새로 시작, 선호는 재사용")

        other = await graph.ainvoke({}, t3, context=b)
        assert other["answer"] == "아직 저장한 선호 없음"
        print("PASS 다른 사용자 namespace: 선호 없음")

        before = await store.aget(("users", a.user_id), "preferences")
        await store.aput(("users", a.user_id), "preferences", {"text": UPDATED})
        after = await store.aget(("users", a.user_id), "preferences")
        assert after.value["text"] == UPDATED
        assert before.created_at == after.created_at
        assert len(list(store_root.rglob("*.json"))) == 1
        hits = await store.asearch(("users", a.user_id), query="코드 식별자는")
        assert hits and hits[0].value["text"] == UPDATED
        print("PASS 같은 key 업데이트: 원본 파일 1개, 검색 결과 확인")
    finally:
        await store.aclose()

    # 메모리 객체/체크포인터를 새로 만들지만 파일 경로는 유지합니다.
    reopened = MemtomemBaseStore(root=store_root, embedding=embedding)
    try:
        graph = make_graph(reopened)
        result = await graph.ainvoke({}, t1, context=a)
        assert result["visits"] == 1 and result["answer"] == UPDATED
        print("PASS 새 store + 새 checkpointer: 파일 기억만 복원")
    finally:
        await reopened.aclose()

with isolated_lab() as lab_root:
    await demonstrate(lab_root)
assert not lab_root.exists()
print("PASS 임시 상태 정리")


## 4. 결과 해석과 복구

위 코드에서 `PASS` 6줄이 출력되면 성공입니다. 특히 마지막 재개방 검사는 **새 객체에서 파일 기억이 복원됨**을 입증하며,
이전 대화의 체크포인트 복구나 별도 프로세스 재시작을 입증하지 않습니다. 이 실습은 저장소 객체를 닫고 다시 열어 확인합니다.

- import 실패: 안내된 환경에 설치했는지, Jupyter 커널의 Python이 같은지 확인하세요.
- 선호 없음: `user_id`, namespace와 key를 먼저 비교하세요.
- 동기 메서드 오류: Jupyter에서는 `await aput/aget/asearch/aclose`를 사용하세요.
- 임시 파일은 실습 종료 시 삭제됩니다. 실제 앱에서는 별도 영속 경로와 접근 통제를 설계해야 합니다.

### 직접 바꾸기
`PREFERENCE`를 바꿔 Run All을 다시 실행하세요. 새 thread에서도 변경한 문장이 조회되어야 합니다.
**연습:** 사용자 B의 선호도 저장하고 A와 B가 서로 다른 문장을 받는지 검사하세요.
**힌트:** 서로 다른 thread ID와 `Context(user_id=...)`를 함께 사용합니다.

다음은 같은 폴더의 `06_langgraph_retrieval_memory.ipynb`입니다.
[LangGraph 공식 메모리 설명](https://docs.langchain.com/oss/python/langgraph/add-memory)